# Sistemas de almacenamiento y recuperación

Un índice ANN acelera la búsqueda de vecinos próximos: HNSW o IVF organizan los vectores para reducir el trabajo de una consulta. Una **base de datos vectorial** incorpora un sistema alrededor de esa búsqueda: persistencia, identificadores, inserción, actualización, borrado, metadatos, filtros, concurrencia y operación. Las garantías concretas dependen del producto y su configuración. Pinecone gestiona la infraestructura; el código sigue necesitando diseñar el modelo de registro y el ciclo de vida de los datos.

El registro conceptual es `{id, embedding, content, metadata}`. El vector representa el significado; el contenido permite interpretar el resultado; los metadatos determinan su elegibilidad. El ID mantiene la identidad del registro a lo largo de los cambios. LangChain convierte el contenido en embeddings y ofrece una interfaz de vector store; Pinecone almacena y consulta los registros.

Se utiliza un corpus ficticio de documentación empresarial. Sus cifras son reglas inventadas de una organización, no normativa laboral. La carga pequeña permite observar cambios y filtros sin evaluar escala ANN. No se necesita Docker. Las operaciones remotas usan OpenAI y Pinecone y pueden generar consumo; repetir una ingesta con el mismo ID no duplica registros, pero vuelve a generar embeddings.


## Configuración y estados de ejecución

Las claves se leen de `.env` o del entorno. `text-embedding-3-small` se configura con **1536 dimensiones** y el índice con métrica coseno. El nombre del índice debe contener entre 1 y 45 caracteres alfanuméricos en minúscula o guiones, con extremos alfanuméricos.

Si falta una clave, el corpus y las tablas conceptuales permanecen disponibles y las operaciones remotas se omiten. Un error remoto se comunica sin imprimir credenciales ni el mensaje del proveedor. Tras corregir la configuración, se pueden volver a ejecutar las celdas desde esta sección. No hay una sustitución silenciosa por embeddings ficticios.

El índice se reutiliza por nombre. Los namespaces con prefijo `s5-` son espacios dedicados a estos ejemplos; no deben contener datos ajenos. El notebook conserva los registros finales y nunca elimina un índice completo.


In [ ]:
import os
import re
import time
from datetime import date
from copy import deepcopy
from pathlib import Path

import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.vectorstores import VectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

load_dotenv(Path.cwd() / ".env", override=False)
pd.set_option("display.max_colwidth", 100)

In [ ]:

INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "s5-storage-retrieval")
CLOUD = os.getenv("PINECONE_CLOUD", "aws")
REGION = os.getenv("PINECONE_REGION", "us-east-1")
NAMESPACE = "s5-policies-main"
EMBEDDING_MODEL = "text-embedding-3-small"
DIMENSION = 1536
missing_keys = [name for name in ("OPENAI_API_KEY", "PINECONE_API_KEY") if not os.getenv(name, "").strip()]
embeddings = pc = index = vector_store = None
corpus_ready = False

print("Claves pendientes:", ", ".join(missing_keys) if missing_keys else "ninguna")

Claves pendientes: ninguna


In [ ]:
def remote_operation(label, operation, default=None):
    if missing_keys:
        print(f"{label}: omitido. Configura {', '.join(missing_keys)} en .env y vuelve a ejecutar desde configuración.")
        return default
    try:
        return operation()
    except Exception as error:
        print(f"{label}: no completado ({type(error).__name__}). Comprueba claves, permisos, cuota, red y configuración; vuelve a ejecutar la operación.")
        return default


def result_frame(results, method="búsqueda"):
    rows = []
    for rank, result in enumerate(results, 1):
        document, score = result if isinstance(result, tuple) else (result, None)
        rows.append({"método": method, "posición": rank, "score": score,
                     "contenido": document.page_content, **document.metadata})
    return pd.DataFrame(rows, columns=None if rows else ["método", "posición", "score", "contenido", "doc_id", "country", "status", "employee_type"])


def ready_store(store, label):
    if store is None:
        print(f"{label}: omitido. Hace falta una conexión válida al índice.")
        return False
    return True

## Corpus y contrato de metadatos

Se construyen **128 documentos** de forma determinista: ocho políticas, cuatro países, dos tipos de contrato y dos versiones. Las versiones históricas comparten gran parte del vocabulario con las vigentes. Esto hace que una búsqueda semántica pueda encontrar una política plausible pero inaplicable.

Los metadatos son planos: strings y números, sin valores nulos ni estructuras anidadas. `date` permite lectura humana; `effective_day` representa la misma fecha como ordinal para filtros de rango. `employee_type`, `status`, `country` y `access_level` son atributos distintos. `source` identifica el origen y `version` la versión del contenido. `tenant` identifica la organización, aunque la separación entre organizaciones se realizará también mediante namespaces.

Cada documento es una unidad recuperable completa. Fragmentar textos largos exigiría IDs por fragmento y conservar un identificador de documento padre; aquí no hace falta un splitter.


In [ ]:
countries = {
    "ES": "España",
    "MX": "México",
    "FR": "Francia",
    "DE": "Alemania",
}
policy_texts = {
    "vacaciones": "Las vacaciones se solicitan en el portal de personas. La solicitud requiere aprobación del responsable y cobertura del equipo. El saldo se consulta antes de reservar viajes.",
    "teletrabajo": "El trabajo remoto requiere un acuerdo registrado y disponibilidad durante la franja compartida del equipo. Los dispositivos deben usar VPN y bloqueo de pantalla.",
    "gastos": "Los gastos de desplazamiento se presentan con factura, centro de coste y motivo del viaje. Finanzas revisa el expediente y devuelve solicitudes incompletas.",
    "seguridad": "Los incidentes de seguridad se comunican al equipo de respuesta mediante el canal de emergencia. No se borran evidencias y se revocan credenciales comprometidas.",
    "compras": "Las compras requieren una solicitud y validación del presupuesto. Los nuevos proveedores pasan una revisión de riesgos antes de emitir una orden.",
    "onboarding": "La incorporación incluye verificación de identidad, entrega de equipo y alta de permisos por rol. El responsable confirma la lista de accesos necesaria.",
    "datos": "Los datos de clientes se clasifican antes de compartirlos. Los ficheros confidenciales se almacenan en repositorios autorizados y se limita el acceso al mínimo necesario.",
    "formacion": "Las actividades de formación requieren objetivo profesional, presupuesto y autorización del responsable. Al finalizar se registra la certificación y el gasto asociado.",
}

documents = []
for category, body in policy_texts.items():
    for country, country_name in countries.items():
        for employee_type in ("internal", "contractor"):
            for version in (1, 2):
                doc_id = f"{category}-{country.lower()}-{employee_type}-v{version}"
                effective_date = date(2025 if version == 1 else 2026, 1, 1)
                audience = "personal interno" if employee_type == "internal" else "colaboradores externos"
                detail = (f"El calendario corporativo reserva {24 + list(countries).index(country) + version} días para el personal interno."
                          if category == "vacaciones" and employee_type == "internal"
                          else "Las ausencias de colaboradores externos se coordinan según contrato; no se asigna saldo corporativo."
                          if category == "vacaciones" else f"La tramitación corresponde a la oficina de {country_name}.")
                metadata = {
                    "doc_id": doc_id, "language": "es", "country": country,
                    "category": category, "date": effective_date.isoformat(),
                    "effective_day": effective_date.toordinal(), "employee_type": employee_type,
                    "access_level": "restricted" if category in ("seguridad", "datos") else "standard",
                    "source": f"intranet://acme/{country.lower()}/{category}",
                    "version": version, "status": "active" if version == 2 else "archived",
                    "tenant": "acme", "embedding_model": EMBEDDING_MODEL,
                }
                content = f"Política de {category} — {country_name}. Destinatarios: {audience}. Versión {version}. {body} {detail}"
                documents.append(Document(id=doc_id, page_content=content, metadata=metadata))
                
document_ids = [document.metadata["doc_id"] for document in documents]
corpus_frame = pd.DataFrame([{"contenido": document.page_content, **document.metadata} for document in documents])
assert len(documents) == 128 and len(set(document_ids)) == 128
assert all("text" not in document.metadata for document in documents)
display(corpus_frame)


,contenido,doc_id,language,country,category,date,effective_day,employee_type,access_level,source,version,status,tenant,embedding_model
0,Política de vacaciones — España. Destinatarios: personal interno. Versión 1. Las vacaciones se s...,vacaciones-es-internal-v1,es,ES,vacaciones,2025-01-01,739252,internal,standard,intranet://acme/es/vacaciones,1,archived,acme,text-embedding-3-small
1,Política de vacaciones — España. Destinatarios: personal interno. Versión 2. Las vacaciones se s...,vacaciones-es-internal-v2,es,ES,vacaciones,2026-01-01,739617,internal,standard,intranet://acme/es/vacaciones,2,active,acme,text-embedding-3-small
2,Política de vacaciones — España. Destinatarios: colaboradores externos. Versión 1. Las vacacione...,vacaciones-es-contractor-v1,es,ES,vacaciones,2025-01-01,739252,contractor,standard,intranet://acme/es/vacaciones,1,archived,acme,text-embedding-3-small
3,Política de vacaciones — España. Destinatarios: colaboradores externos. Versión 2. Las vacacione...,vacaciones-es-contractor-v2,es,ES,vacaciones,2026-01-01,739617,contractor,standard,intranet://acme/es/vacaciones,2,active,acme,text-embedding-3-small
4,Política de vacaciones — México. Destinatarios: personal interno. Versión 1. Las vacaciones se s...,vacaciones-mx-internal-v1,es,MX,vacaciones,2025-01-01,739252,internal,standard,intranet://acme/mx/vacaciones,1,archived,acme,text-embedding-3-small
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,Política de formacion — Francia. Destinatarios: colaboradores externos. Versión 2. Las actividad...,formacion-fr-contractor-v2,es,FR,formacion,2026-01-01,739617,contractor,standard,intranet://acme/fr/formacion,2,active,acme,text-embedding-3-small
124,Política de formacion — Alemania. Destinatarios: personal interno. Versión 1. Las actividades de...,formacion-de-internal-v1,es,DE,formacion,2025-01-01,739252,internal,standard,intranet://acme/de/formacion,1,archived,acme,text-embedding-3-small
125,Política de formacion — Alemania. Destinatarios: personal interno. Versión 2. Las actividades de...,formacion-de-internal-v2,es,DE,formacion,2026-01-01,739617,internal,standard,intranet://acme/de/formacion,2,active,acme,text-embedding-3-small
126,Política de formacion — Alemania. Destinatarios: colaboradores externos. Versión 1. Las activida...,formacion-de-contractor-v1,es,DE,formacion,2025-01-01,739252,contractor,standard,intranet://acme/de/formacion,1,archived,acme,text-embedding-3-small


In [3]:
display(corpus_frame.groupby(["country", "employee_type", "status"]).size().rename("documentos").reset_index())
example_record = {
    "id": document_ids[0],
    "embedding": "vector de 1536 floats, calculado en la ingesta",
    "content": documents[0].page_content,
    "metadata": documents[0].metadata,
}
display(pd.DataFrame([example_record]))


,country,employee_type,status,documentos
0,DE,contractor,active,8
1,DE,contractor,archived,8
2,DE,internal,active,8
3,DE,internal,archived,8
4,ES,contractor,active,8
5,ES,contractor,archived,8
6,ES,internal,active,8
7,ES,internal,archived,8
8,FR,contractor,active,8
9,FR,contractor,archived,8


,id,embedding,content,metadata
0,vacaciones-es-internal-v1,"vector de 1536 floats, calculado en la ingesta",Política de vacaciones — España. Destinatarios: personal interno. Versión 1. Las vacaciones se s...,"{'doc_id': 'vacaciones-es-internal-v1', 'language': 'es', 'country': 'ES', 'category': 'vacacion..."


## Embeddings y configuración específica de Pinecone

`OpenAIEmbeddings` implementa la interfaz de embeddings de LangChain. `PineconeVectorStore.add_documents` genera los vectores y hace upsert; el contenido se conserva en el campo de metadatos `text`. Ese nombre queda reservado para el adaptador.

La creación y descripción del índice corresponden al SDK de Pinecone, porque `VectorStore` no es una abstracción de administración de infraestructura. Se comprueban dimensión y métrica antes de reutilizar el recurso. Que dos modelos tengan la misma dimensión no significa que sus espacios sean compatibles: el índice debe dedicarse al modelo configurado, y `embedding_model` permite auditar los registros.

La región y nube se pueden modificar en `.env` si el plan no admite las predeterminadas. La espera de disponibilidad tiene un límite. Las escrituras de Pinecone son eventualmente consistentes; una confirmación de upsert no garantiza visibilidad inmediata en fetch o búsqueda. La siguiente función comprueba los IDs mediante fetch y su presencia en una consulta vectorial, sin asumir una pausa fija suficiente.

Fuentes: [integración LangChain](https://docs.langchain.com/oss/python/integrations/vectorstores/pinecone), [consistencia de Pinecone](https://docs.pinecone.io/guides/index-data/data-ingestion-overview).


In [ ]:
def connect_pinecone():

    if not re.fullmatch(r"[a-z0-9](?:[a-z0-9-]{0,43}[a-z0-9])?", INDEX_NAME):
        raise ValueError("PINECONE_INDEX_NAME debe tener 1–45 caracteres: minúsculas, números y guiones, con extremos alfanuméricos.")

    client = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

    if not client.has_index(INDEX_NAME):
        client.create_index(name=INDEX_NAME, dimension=DIMENSION, metric="cosine",
                            spec=ServerlessSpec(cloud=CLOUD, region=REGION))

    deadline = time.monotonic() + 90
    while True:
        description = client.describe_index(INDEX_NAME)
        if description.dimension != DIMENSION or description.metric != "cosine":
            raise ValueError("El índice existente tiene dimensión o métrica incompatible; configura otro nombre.")
        if description.status["ready"]:
            break
        if time.monotonic() >= deadline:
            raise TimeoutError("El índice todavía no está listo.")
        time.sleep(2)

    model = OpenAIEmbeddings(model=EMBEDDING_MODEL, dimensions=DIMENSION, max_retries=2)
    connected_index = client.Index(host=description.host)
    store = PineconeVectorStore(index=connected_index, embedding=model, namespace=NAMESPACE, text_key="text")

    return client, connected_index, model, store

connection = remote_operation("Conectar/crear índice", connect_pinecone)
if connection is not None:
    pc, index, embeddings, vector_store = connection
    print(f"Índice conectado: {INDEX_NAME}; dimensión={DIMENSION}; métrica=cosine; namespace={NAMESPACE}")


Índice conectado: s5-storage-retrieval; dimensión=1536; métrica=cosine; namespace=s5-policies-main


In [ ]:
def wait_for_records(namespace, expected_documents=(), absent_ids=(), timeout=45):

    expected = {doc.metadata["doc_id"]: doc for doc in expected_documents}
    all_ids = list(expected) + list(absent_ids)

    if not all_ids:
        return True

    probe_vector = embeddings.embed_query(expected_documents[0].page_content) if expected else None
    deadline = time.monotonic() + timeout

    while time.monotonic() < deadline:
        fetched = {}
        for start in range(0, len(all_ids), 100):
            fetched.update(index.fetch(ids=all_ids[start:start + 100], namespace=namespace).vectors)

        content_visible = all(
            doc_id in fetched and fetched[doc_id].metadata.get("text") == doc.page_content
            and all(fetched[doc_id].metadata.get(key) == value for key, value in doc.metadata.items())
            for doc_id, doc in expected.items()
        )
        deletion_visible = all(doc_id not in fetched for doc_id in absent_ids)
        query_visible = True

        if expected and content_visible:
            matches = index.query(namespace=namespace, vector=probe_vector,
                                  top_k=max(200, len(expected)), include_metadata=True).matches
            queried = {match.id: match.metadata for match in matches}
            query_visible = all(doc_id in queried and queried[doc_id].get("text") == doc.page_content
                                and all(queried[doc_id].get(key) == value for key, value in doc.metadata.items())
                                for doc_id, doc in expected.items())

        if content_visible and deletion_visible and query_visible:
            return True

        time.sleep(2)

    raise TimeoutError("Los cambios no son visibles todavía. Reintenta más tarde; no se ha confirmado la lectura.")


def ingest_corpus():
    vector_store.add_documents(documents=deepcopy(documents), ids=document_ids, batch_size=32)
    wait_for_records(NAMESPACE, documents)
    return True

if ready_store(vector_store, "Ingesta"):
    corpus_ready = remote_operation("Ingesta y comprobación de visibilidad", ingest_corpus, False)
    if corpus_ready:
        print(f"{len(document_ids)} registros visibles. IDs explícitos: upsert reproducible.")


128 registros visibles. IDs explícitos: upsert reproducible.


## Recuperación semántica y elegibilidad

Una consulta busca cercanía entre el embedding de la pregunta y los embeddings almacenados. El score devuelto aquí es la similitud del backend con métrica coseno; no es una probabilidad de que el contenido sea correcto, ni un score comparable sin más con otros proveedores.

La consulta «política de vacaciones para empleados» no especifica país, contrato ni vigencia. Es razonable que recupere variantes históricas o de otro país. El filtro define qué registros pueden entrar en el conjunto de resultados; la similitud ordena ese conjunto. Una política muy relevante en significado puede ser inelegible para el solicitante.

Los filtros enviados por LangChain usan la sintaxis de Pinecone. Un ejemplo combina `country == ES`, `status == active` y `employee_type == internal`. Otros ejemplos usan pertenencia (`$in`), rango (`$gte`) y alternativas (`$or`). Un filtro de `access_level` solo es una condición de consulta: en una aplicación debe construirse desde una identidad autorizada, sin aceptar permisos arbitrarios del usuario.

Fuente: [operadores de filtros Pinecone](https://docs.pinecone.io/guides/search/filter-by-metadata).


In [ ]:
query = "¿Cuál es la política de vacaciones para empleados?"

eligible_filter = {
    "$and": [
        {"country": {"$eq": "ES"}},
        {"status": {"$eq": "active"}},
        {"employee_type": {"$eq": "internal"}},
    ]
}

unfiltered_results = filtered_results = []
if corpus_ready:
    plain_documents = remote_operation("Búsqueda sin score", lambda: vector_store.similarity_search(query, k=4), [])
    display(result_frame(plain_documents, "similarity_search"))
    unfiltered_results = remote_operation("Búsqueda sin filtros", lambda: vector_store.similarity_search_with_score(query, k=6), [])
    filtered_results = remote_operation("Búsqueda con filtros", lambda: vector_store.similarity_search_with_score(query, k=6, filter=eligible_filter), [])
else:
    print("Búsquedas omitidas: la ingesta no está confirmada.")
display(pd.concat([result_frame(unfiltered_results, "sin filtro"), result_frame(filtered_results, "filtro en base")], ignore_index=True))


,método,posición,score,contenido,access_level,category,country,date,doc_id,effective_day,embedding_model,employee_type,language,source,status,tenant,version
0,similarity_search,1,None,Política de vacaciones — México. Destinatarios: personal interno. Versión 2. Las vacaciones se s...,standard,vacaciones,MX,2026-01-01,vacaciones-mx-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/mx/vacaciones,active,acme,2.0
1,similarity_search,2,None,Política de vacaciones — México. Destinatarios: personal interno. Versión 1. Las vacaciones se s...,standard,vacaciones,MX,2025-01-01,vacaciones-mx-internal-v1,739252.0,text-embedding-3-small,internal,es,intranet://acme/mx/vacaciones,archived,acme,1.0
2,similarity_search,3,None,Política de vacaciones — España. Destinatarios: personal interno. Versión 2. Las vacaciones se s...,standard,vacaciones,ES,2026-01-01,vacaciones-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/vacaciones,active,acme,2.0
3,similarity_search,4,None,Política de vacaciones — España. Destinatarios: personal interno. Versión 1. Las vacaciones se s...,standard,vacaciones,ES,2025-01-01,vacaciones-es-internal-v1,739252.0,text-embedding-3-small,internal,es,intranet://acme/es/vacaciones,archived,acme,1.0


,método,posición,score,contenido,access_level,category,country,date,doc_id,effective_day,embedding_model,employee_type,language,source,status,tenant,version
0,sin filtro,1,0.762113,Política de vacaciones — México. Destinatarios: personal interno. Versión 2. Las vacaciones se s...,standard,vacaciones,MX,2026-01-01,vacaciones-mx-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/mx/vacaciones,active,acme,2.0
1,sin filtro,2,0.761841,Política de vacaciones — México. Destinatarios: personal interno. Versión 1. Las vacaciones se s...,standard,vacaciones,MX,2025-01-01,vacaciones-mx-internal-v1,739252.0,text-embedding-3-small,internal,es,intranet://acme/mx/vacaciones,archived,acme,1.0
2,sin filtro,3,0.761427,Política de vacaciones — España. Destinatarios: personal interno. Versión 2. Las vacaciones se s...,standard,vacaciones,ES,2026-01-01,vacaciones-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/vacaciones,active,acme,2.0
3,sin filtro,4,0.760342,Política de vacaciones — España. Destinatarios: personal interno. Versión 1. Las vacaciones se s...,standard,vacaciones,ES,2025-01-01,vacaciones-es-internal-v1,739252.0,text-embedding-3-small,internal,es,intranet://acme/es/vacaciones,archived,acme,1.0
4,sin filtro,5,0.745528,Política de vacaciones — México. Destinatarios: colaboradores externos. Versión 1. Las vacacione...,standard,vacaciones,MX,2025-01-01,vacaciones-mx-contractor-v1,739252.0,text-embedding-3-small,contractor,es,intranet://acme/mx/vacaciones,archived,acme,1.0
5,sin filtro,6,0.744735,Política de vacaciones — España. Destinatarios: colaboradores externos. Versión 1. Las vacacione...,standard,vacaciones,ES,2025-01-01,vacaciones-es-contractor-v1,739252.0,text-embedding-3-small,contractor,es,intranet://acme/es/vacaciones,archived,acme,1.0
6,filtro en base,1,0.761427,Política de vacaciones — España. Destinatarios: personal interno. Versión 2. Las vacaciones se s...,standard,vacaciones,ES,2026-01-01,vacaciones-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/vacaciones,active,acme,2.0
7,filtro en base,2,0.530394,Política de teletrabajo — España. Destinatarios: personal interno. Versión 2. El trabajo remoto ...,standard,teletrabajo,ES,2026-01-01,teletrabajo-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/teletrabajo,active,acme,2.0
8,filtro en base,3,0.430681,Política de onboarding — España. Destinatarios: personal interno. Versión 2. La incorporación in...,standard,onboarding,ES,2026-01-01,onboarding-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/onboarding,active,acme,2.0
9,filtro en base,4,0.417420,Política de gastos — España. Destinatarios: personal interno. Versión 2. Los gastos de desplazam...,standard,gastos,ES,2026-01-01,gastos-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/gastos,active,acme,2.0


In [7]:
filter_examples = {
    "países y vigencia": {"$and": [{"country": {"$in": ["ES", "MX"]}}, {"status": {"$eq": "active"}}]},
    "fecha y categoría": {"$and": [{"effective_day": {"$gte": date(2026, 1, 1).toordinal()}}, {"category": {"$eq": "vacaciones"}}]},
    "categorías alternativas": {"$and": [{"status": {"$eq": "active"}}, {"$or": [{"category": {"$eq": "teletrabajo"}}, {"category": {"$eq": "seguridad"}}]}]},
    "acceso estándar": {"$and": [{"access_level": {"$eq": "standard"}}, {"country": {"$eq": "ES"}}]},
}
display(pd.DataFrame([{"ejemplo": label, "filtro": expression} for label, expression in filter_examples.items()]))
if corpus_ready:
    frames = []
    for label, expression in filter_examples.items():
        results = remote_operation(label, lambda expression=expression: vector_store.similarity_search_with_score(query, k=3, filter=expression), [])
        frames.append(result_frame(results, label))
    display(pd.concat(frames, ignore_index=True))


,ejemplo,filtro
0,países y vigencia,"{'$and': [{'country': {'$in': ['ES', 'MX']}}, {'status': {'$eq': 'active'}}]}"
1,fecha y categoría,"{'$and': [{'effective_day': {'$gte': 739617}}, {'category': {'$eq': 'vacaciones'}}]}"
2,categorías alternativas,"{'$and': [{'status': {'$eq': 'active'}}, {'$or': [{'category': {'$eq': 'teletrabajo'}}, {'catego..."
3,acceso estándar,"{'$and': [{'access_level': {'$eq': 'standard'}}, {'country': {'$eq': 'ES'}}]}"


,método,posición,score,contenido,access_level,category,country,date,doc_id,effective_day,embedding_model,employee_type,language,source,status,tenant,version
0,países y vigencia,1,0.762113,Política de vacaciones — México. Destinatarios: personal interno. Versión 2. Las vacaciones se s...,standard,vacaciones,MX,2026-01-01,vacaciones-mx-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/mx/vacaciones,active,acme,2.0
1,países y vigencia,2,0.761427,Política de vacaciones — España. Destinatarios: personal interno. Versión 2. Las vacaciones se s...,standard,vacaciones,ES,2026-01-01,vacaciones-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/vacaciones,active,acme,2.0
2,países y vigencia,3,0.744685,Política de vacaciones — México. Destinatarios: colaboradores externos. Versión 2. Las vacacione...,standard,vacaciones,MX,2026-01-01,vacaciones-mx-contractor-v2,739617.0,text-embedding-3-small,contractor,es,intranet://acme/mx/vacaciones,active,acme,2.0
3,fecha y categoría,1,0.762113,Política de vacaciones — México. Destinatarios: personal interno. Versión 2. Las vacaciones se s...,standard,vacaciones,MX,2026-01-01,vacaciones-mx-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/mx/vacaciones,active,acme,2.0
4,fecha y categoría,2,0.761427,Política de vacaciones — España. Destinatarios: personal interno. Versión 2. Las vacaciones se s...,standard,vacaciones,ES,2026-01-01,vacaciones-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/vacaciones,active,acme,2.0
5,fecha y categoría,3,0.744685,Política de vacaciones — México. Destinatarios: colaboradores externos. Versión 2. Las vacacione...,standard,vacaciones,MX,2026-01-01,vacaciones-mx-contractor-v2,739617.0,text-embedding-3-small,contractor,es,intranet://acme/mx/vacaciones,active,acme,2.0
6,categorías alternativas,1,0.530394,Política de teletrabajo — España. Destinatarios: personal interno. Versión 2. El trabajo remoto ...,standard,teletrabajo,ES,2026-01-01,teletrabajo-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/teletrabajo,active,acme,2.0
7,categorías alternativas,2,0.526061,Política de teletrabajo — México. Destinatarios: colaboradores externos. Versión 2. El trabajo r...,standard,teletrabajo,MX,2026-01-01,teletrabajo-mx-contractor-v2,739617.0,text-embedding-3-small,contractor,es,intranet://acme/mx/teletrabajo,active,acme,2.0
8,categorías alternativas,3,0.525899,Política de teletrabajo — México. Destinatarios: personal interno. Versión 2. El trabajo remoto ...,standard,teletrabajo,MX,2026-01-01,teletrabajo-mx-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/mx/teletrabajo,active,acme,2.0
9,acceso estándar,1,0.761427,Política de vacaciones — España. Destinatarios: personal interno. Versión 2. Las vacaciones se s...,standard,vacaciones,ES,2026-01-01,vacaciones-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/vacaciones,active,acme,2.0


## Filtrar antes de limitar frente a filtrar después

Conceptualmente, **pre-filtering** restringe candidatos elegibles antes de la selección final; **post-filtering** elimina resultados después de obtener un Top-k sin restricciones. En búsqueda aproximada, cómo se integra un filtro con el recorrido del índice afecta a rendimiento y recall. No se presupone un algoritmo interno específico de Pinecone: aquí se observa su contrato de filtrado integrado en la consulta.

El post-filtro manual pide 12 candidatos y después conserva los que cumplen tres condiciones. Puede devolver menos de los seis resultados deseados, aunque existan suficientes registros elegibles en el corpus. Aumentar el Top-k reduce ese riesgo sin garantizarlo con un límite finito. También transfiere candidatos inelegibles al cliente.

La comparación depende del ranking real: no se fuerza artificialmente un déficit. La tabla incluye el total elegible del corpus para distinguir falta de candidatos de falta de documentos. La búsqueda integrada tampoco garantiza completar k si hay pocos registros elegibles.


In [8]:
def is_eligible(document):
    metadata = document.metadata
    return metadata.get("country") == "ES" and metadata.get("status") == "active" and metadata.get("employee_type") == "internal"

desired_k = 6
eligible_total = sum(is_eligible(document) for document in documents)
if corpus_ready:
    candidates = remote_operation("Top-12 sin filtro", lambda: vector_store.similarity_search_with_score(query, k=12), [])
    manual_results = [(doc, score) for doc, score in candidates if is_eligible(doc)][:desired_k]
    integrated_results = remote_operation("Filtro integrado", lambda: vector_store.similarity_search_with_score(query, k=desired_k, filter=eligible_filter), [])
    display(pd.DataFrame([
        {"método": "post-filtro Python", "candidatos recibidos": len(candidates), "resultados": len(manual_results), "k deseado": desired_k, "elegibles en corpus": eligible_total},
        {"método": "filtro en base", "candidatos recibidos": len(integrated_results), "resultados": len(integrated_results), "k deseado": desired_k, "elegibles en corpus": eligible_total},
    ]))
    display(pd.concat([result_frame(manual_results, "post-filtro"), result_frame(integrated_results, "filtro en base")], ignore_index=True))
else:
    print(f"Comparación remota omitida. El corpus contiene {eligible_total} documentos elegibles.")


,método,candidatos recibidos,resultados,k deseado,elegibles en corpus
0,post-filtro Python,12,1,6,8
1,filtro en base,6,6,6,8


,método,posición,score,contenido,access_level,category,country,date,doc_id,effective_day,embedding_model,employee_type,language,source,status,tenant,version
0,post-filtro,1,0.761427,Política de vacaciones — España. Destinatarios: personal interno. Versión 2. Las vacaciones se s...,standard,vacaciones,ES,2026-01-01,vacaciones-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/vacaciones,active,acme,2.0
1,filtro en base,1,0.761427,Política de vacaciones — España. Destinatarios: personal interno. Versión 2. Las vacaciones se s...,standard,vacaciones,ES,2026-01-01,vacaciones-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/vacaciones,active,acme,2.0
2,filtro en base,2,0.530394,Política de teletrabajo — España. Destinatarios: personal interno. Versión 2. El trabajo remoto ...,standard,teletrabajo,ES,2026-01-01,teletrabajo-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/teletrabajo,active,acme,2.0
3,filtro en base,3,0.430681,Política de onboarding — España. Destinatarios: personal interno. Versión 2. La incorporación in...,standard,onboarding,ES,2026-01-01,onboarding-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/onboarding,active,acme,2.0
4,filtro en base,4,0.417420,Política de gastos — España. Destinatarios: personal interno. Versión 2. Los gastos de desplazam...,standard,gastos,ES,2026-01-01,gastos-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/gastos,active,acme,2.0
5,filtro en base,5,0.398829,Política de formacion — España. Destinatarios: personal interno. Versión 2. Las actividades de f...,standard,formacion,ES,2026-01-01,formacion-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/formacion,active,acme,2.0
6,filtro en base,6,0.369389,Política de datos — España. Destinatarios: personal interno. Versión 2. Los datos de clientes se...,restricted,datos,ES,2026-01-01,datos-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/datos,active,acme,2.0


## CRUD: identidad, contenido y metadatos

Insertar un nuevo ID añade un registro. Reutilizar un ID hace upsert y reemplaza su vector y metadatos: sirve para cambiar contenido con re-embedding sin inventar un método `update_document` en LangChain. La actualización de solo metadatos usa `index.update(set_metadata=...)` del cliente Pinecone y no requiere recalcular el vector; añade o cambia los campos indicados, no reemplaza todo el diccionario.

Se crea un procedimiento temporal, se modifica contenido y versión, se cambia su nivel de acceso mediante el SDK y finalmente se borra por ID mediante LangChain. Cada lectura comprueba la visibilidad del cambio. El borrado se verifica con fetch; debido a consistencia eventual, una consulta inmediatamente posterior podría seguir viendo el registro. La secuencia puede repetirse: solo opera sobre su ID explícito en el namespace dedicado.

Fuentes: [actualización Pinecone](https://docs.pinecone.io/guides/manage-data/update-data), [CRUD LangChain](https://docs.langchain.com/oss/python/integrations/vectorstores/pinecone).


In [9]:
temporary_metadata = deepcopy(next(doc.metadata for doc in documents if doc.metadata["doc_id"] == "vacaciones-es-internal-v2"))
temporary_metadata.update(doc_id="crud-vacaciones-es", source="intranet://acme/es/ausencias-urgentes", version=1)
new_document = Document(id=temporary_metadata["doc_id"],
                        page_content="Procedimiento de vacaciones urgentes en España: registrar la ausencia en el portal y avisar al responsable antes de las 09:00.",
                        metadata=temporary_metadata)
crud_id = new_document.metadata["doc_id"]


def create_record():
    vector_store.add_documents(documents=deepcopy([new_document]), ids=[crud_id])
    wait_for_records(NAMESPACE, [new_document])
    return vector_store.similarity_search_with_score("vacaciones urgentes", k=2, filter={"doc_id": {"$eq": crud_id}})

crud_created = False
if corpus_ready:
    created_results = remote_operation("Crear registro", create_record)
    crud_created = created_results is not None
    if crud_created:
        display(result_frame(created_results, "insertado"))
else:
    print("CRUD omitido: la ingesta no está confirmada.")


,método,posición,score,contenido,access_level,category,country,date,doc_id,effective_day,embedding_model,employee_type,language,source,status,tenant,version
0,insertado,1,0.579256,Procedimiento de vacaciones urgentes en España: registrar la ausencia en el portal y avisar al r...,standard,vacaciones,ES,2026-01-01,crud-vacaciones-es,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/ausencias-urgentes,active,acme,1.0


In [10]:
updated_document = Document(id=crud_id,
    page_content="Procedimiento de vacaciones urgentes en España: registrar la ausencia antes de las 10:00 y adjuntar la aprobación del responsable en el portal.",
    metadata={**temporary_metadata, "version": 2})


def replace_record():
    vector_store.add_documents(documents=deepcopy([updated_document]), ids=[crud_id])
    wait_for_records(NAMESPACE, [updated_document])
    index.update(id=crud_id, namespace=NAMESPACE, set_metadata={"access_level": "restricted", "reviewed_by": "people_ops"})
    expected = Document(id=crud_id, page_content=updated_document.page_content,
                        metadata={**updated_document.metadata, "access_level": "restricted", "reviewed_by": "people_ops"})
    wait_for_records(NAMESPACE, [expected])
    return vector_store.similarity_search_with_score("vacaciones urgentes", k=1, filter={"doc_id": {"$eq": crud_id}})

if crud_created:
    updated_results = remote_operation("Reemplazar contenido y actualizar metadatos", replace_record, [])
    display(result_frame(updated_results, "actualizado"))


,método,posición,score,contenido,access_level,category,country,date,doc_id,effective_day,embedding_model,employee_type,language,reviewed_by,source,status,tenant,version
0,actualizado,1,0.575936,Procedimiento de vacaciones urgentes en España: registrar la ausencia antes de las 10:00 y adjun...,restricted,vacaciones,ES,2026-01-01,crud-vacaciones-es,739617.0,text-embedding-3-small,internal,es,people_ops,intranet://acme/es/ausencias-urgentes,active,acme,2.0


In [11]:
def delete_record():
    vector_store.delete(ids=[crud_id])
    wait_for_records(NAMESPACE, absent_ids=[crud_id])
    return pd.DataFrame([{"id": crud_id, "namespace": NAMESPACE, "presente en fetch": crud_id in index.fetch(ids=[crud_id], namespace=NAMESPACE).vectors}])

if crud_created:
    deletion_result = remote_operation("Eliminar registro temporal", delete_record)
    if deletion_result is not None:
        display(deletion_result)


,id,namespace,presente en fetch
0,crud-vacaciones-es,s5-policies-main,False


## Persistencia y reconexión

El objeto `PineconeVectorStore` contiene un cliente, la configuración del embedding y un namespace; no es el lugar donde viven los registros. Una segunda instancia puede recuperarlos sin llamar a `add_documents`. Reiniciar Python tampoco borra el índice. La permanencia de los datos depende del recurso remoto, su política de retención y las operaciones de borrado.

Se crea otro cliente y se consulta el mismo índice y namespace. La consulta usa un ID del corpus original para que el resultado no dependa de la ordenación de varios documentos similares. Las credenciales deben pertenecer al proyecto que contiene ese índice.


In [12]:
reopened_store = None

def reconnect_and_search():
    global reopened_store
    new_client = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
    host = new_client.describe_index(INDEX_NAME).host
    reopened_store = PineconeVectorStore(index=new_client.Index(host=host), embedding=embeddings,
                                         namespace=NAMESPACE, text_key="text")
    return reopened_store.similarity_search_with_score(query, k=1, filter={"doc_id": {"$eq": "vacaciones-es-internal-v2"}})

if corpus_ready:
    persistent_results = remote_operation("Reconectar sin ingesta", reconnect_and_search, [])
    display(result_frame(persistent_results, "nueva instancia"))


,método,posición,score,contenido,access_level,category,country,date,doc_id,effective_day,embedding_model,employee_type,language,source,status,tenant,version
0,nueva instancia,1,0.761427,Política de vacaciones — España. Destinatarios: personal interno. Versión 2. Las vacaciones se s...,standard,vacaciones,ES,2026-01-01,vacaciones-es-internal-v2,739617.0,text-embedding-3-small,internal,es,intranet://acme/es/vacaciones,active,acme,2.0


## Namespaces y separación entre organizaciones

Un namespace particiona lógicamente los registros de un índice y acota las consultas a esa partición. Los mismos IDs pueden coexistir en namespaces distintos. Los metadatos permiten restricciones más finas dentro de un namespace: país, departamento o estado.

Se crean dos espacios pequeños, `s5-tenant-acme` y `s5-tenant-orbit`, con políticas semánticamente parecidas y reglas distintas. Cada consulta utiliza la instancia asociada a su namespace. El campo `tenant` se conserva para auditoría; no es lo que produce el aislamiento en este ejemplo.

La separación lógica no reemplaza autenticación o autorización. Un servicio con acceso al índice debe seleccionar el namespace según la identidad verificada; no basta con permitir que el cliente envíe un nombre arbitrario.

Fuente: [multi-tenancy en Pinecone](https://docs.pinecone.io/guides/index-data/implement-multitenancy).


In [ ]:
tenant_stores = {}
tenant_rows = []

if corpus_ready:
    for tenant, notice in {"acme": "Las vacaciones se aprueban en PeopleHub con 15 días de antelación.",
                          "orbit": "Las vacaciones se aprueban en OrbitDesk con 7 días de antelación."}.items():
        namespace = f"s5-tenant-{tenant}"
        tenant_documents = [Document(id=f"tenant-policy-{position}", page_content=text,
            metadata={"doc_id": f"tenant-policy-{position}", "tenant": tenant, "country": "ES",
                      "status": "active", "employee_type": "internal", "version": 1,
                      "embedding_model": EMBEDDING_MODEL})
            for position, text in enumerate([notice, f"En {tenant}, las incidencias del portal de personas se notifican al equipo de soporte."], 1)]

        store = PineconeVectorStore(index=index, embedding=embeddings, namespace=namespace)

        def ingest_tenant(store=store, docs=tenant_documents, namespace=namespace):
            store.add_documents(documents=deepcopy(docs), ids=[doc.metadata["doc_id"] for doc in docs])
            wait_for_records(namespace, docs)
            return store.similarity_search_with_score("¿Cómo solicito vacaciones?", k=2)

        results = remote_operation(f"Namespace {namespace}", ingest_tenant)

        if results:
            tenant_stores[tenant] = store
            frame = result_frame(results, namespace)
            assert all(frame["tenant"] == tenant)
            tenant_rows.append(frame)

    if tenant_rows:
        display(pd.concat(tenant_rows, ignore_index=True))
else:
    print("Namespaces omitidos: la ingesta no está confirmada.")


,método,posición,score,contenido,country,doc_id,embedding_model,employee_type,status,tenant,version
0,s5-tenant-acme,1,0.575388,Las vacaciones se aprueban en PeopleHub con 15 días de antelación.,ES,tenant-policy-1,text-embedding-3-small,internal,active,acme,1.0
1,s5-tenant-acme,2,0.258503,"En acme, las incidencias del portal de personas se notifican al equipo de soporte.",ES,tenant-policy-2,text-embedding-3-small,internal,active,acme,1.0
2,s5-tenant-orbit,1,0.565376,Las vacaciones se aprueban en OrbitDesk con 7 días de antelación.,ES,tenant-policy-1,text-embedding-3-small,internal,active,orbit,1.0
3,s5-tenant-orbit,2,0.243936,"En orbit, las incidencias del portal de personas se notifican al equipo de soporte.",ES,tenant-policy-2,text-embedding-3-small,internal,active,orbit,1.0


## Versiones de colección y migración

Separar `s5-policy-v1` y `s5-policy-v2` permite construir un nuevo corpus sin reemplazar el anterior. Un parámetro de la aplicación selecciona la versión a consultar. Esto facilita un patrón de validación paralela o A/B; Pinecone no distribuye tráfico ni evalúa experimentos automáticamente.

Ambas versiones usan aquí el mismo modelo de embeddings. Cambiar de modelo requiere recalcular los embeddings de documentos y queries, construir la nueva versión, verificar calidad y filtros, cambiar el enrutamiento y conservar una ruta de rollback. Si la dimensión cambia se necesita otro índice; aunque no cambie, mezclar modelos en una misma partición puede invalidar la similitud. La limpieza de la versión anterior se realiza después de validar el cambio.

El versionado de contenido de cada registro (`metadata.version`) y el versionado de una colección completa (namespace) resuelven problemas diferentes. El primero identifica una política; el segundo agrupa una entrega coherente del corpus.


In [ ]:
version_stores = {}
version_frames = []

if corpus_ready:
    for version, cutoff in ((1, "09:00"), (2, "10:00")):

        namespace = f"s5-policy-v{version}"

        doc = Document(id="urgent-policy", page_content=f"Política de vacaciones urgentes: registrar la solicitud antes de las {cutoff} y obtener aprobación del responsable.",
            metadata={"doc_id": "urgent-policy", "version": version, "tenant": "acme", "status": "active", "embedding_model": EMBEDDING_MODEL})
        store = PineconeVectorStore(index=index, embedding=embeddings, namespace=namespace)

        def prepare_version(store=store, doc=doc, namespace=namespace):
            store.add_documents(documents=deepcopy([doc]), ids=[doc.metadata["doc_id"]])
            wait_for_records(namespace, [doc])
            return store.similarity_search_with_score("¿Hasta qué hora puedo solicitar vacaciones urgentes?", k=1)

        results = remote_operation(f"Preparar versión {version}", prepare_version)

        if results is not None:
            version_stores[version] = store
            version_frames.append(result_frame(results, namespace))

    if version_frames:
        display(pd.concat(version_frames, ignore_index=True))

active_collection_version = 2
active_store = version_stores.get(active_collection_version)
if active_store is not None:
    display(result_frame(remote_operation("Ruta activa", lambda: active_store.similarity_search(query, k=1), []), f"ruta v{active_collection_version}"))
else:
    print("Ruta activa no disponible: no se confirmó la versión configurada. No se cambia automáticamente a otra versión.")


,método,posición,score,contenido,doc_id,embedding_model,status,tenant,version
0,s5-policy-v1,1,0.703699,Política de vacaciones urgentes: registrar la solicitud antes de las 09:00 y obtener aprobación ...,urgent-policy,text-embedding-3-small,active,acme,1.0
1,s5-policy-v2,1,0.707448,Política de vacaciones urgentes: registrar la solicitud antes de las 10:00 y obtener aprobación ...,urgent-policy,text-embedding-3-small,active,acme,2.0


,método,posición,score,contenido,doc_id,embedding_model,status,tenant,version
0,ruta v2,1,None,Política de vacaciones urgentes: registrar la solicitud antes de las 10:00 y obtener aprobación ...,urgent-policy,text-embedding-3-small,active,acme,2.0
